"""
Testing Script — RNGAP Model (Kumar et al. 2021)
"Multi-class brain tumor classification using residual network
and global average pooling"

Input  : Folder gambar (struktur subfolder per kelas)
Output : Accuracy, Loss, Classification Report, Confusion Matrix
         Precision, Recall, F1-Score per kelas (sesuai Table 3 paper)
         + semua hasil disimpan ke file
""

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
import json
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# ─────────────────────────────────────────────
# KONFIGURASI
# ─────────────────────────────────────────────
TEST_DIR      = "/content/drive/MyDrive/mri_otakv2/Dataset_Split/test"
MODEL_PATH    = "/content/drive/MyDrive/mri_otakv2/Hasil_RNGAPv1/fold_1/best_model_fold1.keras"
CLASS_MAPPING = "/content/drive/MyDrive/mri_otakv2/Hasil_RNGAPv1/class_mapping.json"
OUTPUT_DIR    = "/content/drive/MyDrive/mri_otakv2/Hasil_RNGAPv1/Testing_ResultsFold1"
INPUT_SHAPE   = (224, 224, 3)
BATCH_SIZE    = 2

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# ─────────────────────────────────────────────
# LOAD CLASS MAPPING
# ─────────────────────────────────────────────
def load_class_mapping(mapping_path):
    with open(mapping_path, "r") as f:
        mapping = json.load(f)
    return {int(k): v for k, v in mapping.items()}


# ─────────────────────────────────────────────
# LOAD DATASET TESTING
# ─────────────────────────────────────────────
def load_test_paths(test_dir, class_mapping):
    class_names  = [class_mapping[i] for i in sorted(class_mapping.keys())]
    class_to_idx = {name: idx for idx, name in enumerate(class_names)}

    image_paths, labels = [], []
    for cls_name in class_names:
        cls_dir = os.path.join(test_dir, cls_name)
        if not os.path.isdir(cls_dir):
            print(f"  [PERINGATAN] Folder tidak ditemukan: {cls_dir}")
            continue
        for fname in os.listdir(cls_dir):
            if fname.lower().endswith(('.jpg')):
                image_paths.append(os.path.join(cls_dir, fname))
                labels.append(class_to_idx[cls_name])

    print(f"Total gambar testing: {len(image_paths)}")
    for cls_name in class_names:
        count = labels.count(class_to_idx[cls_name])
        print(f"  {cls_name}: {count} gambar")

    return np.array(image_paths), np.array(labels), class_names

In [ ]:
# ─────────────────────────────────────────────
# BUAT TF DATASET (one-hot — sesuai binary CE)
# ─────────────────────────────────────────────
def make_test_dataset(paths, labels_onehot):
    def preprocess(path, label):
        img = tf.io.read_file(path)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.resize(img, (224, 224))
        img = tf.cast(img, tf.float32)
        img = tf.keras.applications.resnet50.preprocess_input(img)
        return img, label

    dataset = tf.data.Dataset.from_tensor_slices((paths, labels_onehot))
    dataset = dataset.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.batch(2)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset

In [ ]:
# ─────────────────────────────────────────────
# PLOT CONFUSION MATRIX
# ─────────────────────────────────────────────
def plot_confusion_matrix(cm, class_names, save_dir):
    plt.figure(figsize=(max(6, len(class_names) * 2), max(5, len(class_names) * 1.8)))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names,
        linewidths=0.5,
        annot_kws={"size": 13}
    )
    plt.title("Confusion Matrix — RNGAP Model", fontsize=14, fontweight="bold", pad=15)
    plt.ylabel("True Label", fontsize=12)
    plt.xlabel("Predicted Label", fontsize=12)
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    save_path = os.path.join(save_dir, "confusion_matrix.png")
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"Confusion matrix disimpan: {save_path}")

In [ ]:
# TESTING
def main():

    # ── Load class mapping ──────────────────────────────────────
    print(f"\nMemuat class mapping dari: {CLASS_MAPPING}")
    class_mapping = load_class_mapping(CLASS_MAPPING)
    print(f"Kelas: {class_mapping}")

    # ── Load model ──────────────────────────────────────────────
    print(f"\nMemuat model dari: {MODEL_PATH}")
    model = tf.keras.models.load_model(MODEL_PATH)
    print("Model berhasil dimuat.")
    model.summary()

    # ── Load data testing ───────────────────────────────────────
    print(f"\nMemuat data testing dari: {TEST_DIR}")
    image_paths, labels, class_names = load_test_paths(TEST_DIR, class_mapping)
    num_classes = len(class_names)

    # One-hot encode labels
    labels_onehot = tf.keras.utils.to_categorical(labels, num_classes=num_classes)
    test_ds       = make_test_dataset(image_paths, labels_onehot)

    # ── Prediksi ────────────────────────────────────────────────
    print("\nMemprediksi seluruh data testing...")
    pred_probs = model.predict(test_ds, verbose=1)
    pred       = np.argmax(pred_probs, axis=1)
    true       = labels

    #Buat confusion matrix
    cm = confusion_matrix(true, pred)
    plot_confusion_matrix(cm, class_names, test_result)

    # Hitung metrik
    accuracy = accuracy_score(true, pred)
    precision = precision_score(true, pred, zero_division=0, average="macro")
    recall = recall_score(true, pred, zero_division=0, average="macro")
    f1 = f1_score(true, pred, zero_division=0, average="macro")

    #Ringkasan Akhir
    print(f"  Akurasi   : {accuracy:.4f}")
    print(f"  Presisi  : {precision:.4f}")
    print(f"  Recall     : {recall:.4f}")
    print(f"  F1-Score   : {f1:.4f}")


In [ ]:
main()


Memuat class mapping dari: /content/drive/MyDrive/mri_otakv2/Hasil_RNGAPv1/class_mapping.json
Kelas: {0: 'glioma', 1: 'meningioma', 2: 'pituitari'}

Memuat model dari: /content/drive/MyDrive/mri_otakv2/Hasil_RNGAPv1/fold_1/best_model_fold1.keras
Model berhasil dimuat.


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │         6,147 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 47,134,600 (179.80 MB)

 Trainable params: 23,540,739 (89.80 MB)

 Non-trainable params: 53,120 (207.50 KB)

 Optimizer params: 23,540,741 (89.80 MB)


Memuat data testing dari: /content/drive/MyDrive/mri_otakv2/Dataset_Split/test
Total gambar testing: 494
  glioma: 165 gambar
  meningioma: 164 gambar
  pituitari: 165 gambar

Memprediksi seluruh data testing...
247/247 ━━━━━━━━━━━━━━━━━━━━ 7s 14ms/step
Confusion Matrix:
                         glioma    meningioma     pituitari
           glioma           158             7             0
       meningioma             8           155             1
        pituitari             0             4           161


  Akurasi  : 0.9595
  Presisi  : 0.9598
  Recall   : 0.9595
  F1-Score : 0.9596
